# Netflix Content Analysis
Cleaning and analysing the Netflix Movies and TV Shows dataset with Pandas, then merging in IMDb ratings to explore how content is rated.

## 1. Load Data

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv
/kaggle/input/datasets/sukhmandeepsinghbrar/netflix-all-movie-ratings-by-imdb/imdb_data.csv


In [2]:
df = pd.read_csv("/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## 2. Explore the Data

In [3]:
df.shape

(8807, 12)

In [4]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [5]:
df.describe(include='object')

,show_id,type,title,director,cast,country,date_added,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8803,8804,8807,8807
unique,8807,2,8807,4528,7692,748,1767,17,220,514,8775
top,s8807,Movie,Zubaan,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned prope..."
freq,1,6131,1,19,19,2818,109,3207,1793,362,4


## 3. Handle Missing Values
- `director`, `cast`, `country`: filled with 'Unknown' because dropping these rows would have removed a large share of the data (director alone was missing in ~30%).
- `date_added`, `rating`, `duration`: only a handful of rows were missing, so those rows were dropped.

In [6]:
df.director = df.director.fillna('Unknown')

In [7]:
df.cast = df.cast.fillna('Unknown')

In [8]:
df.country = df.country.fillna('Unknown')

In [9]:
df = df.dropna(subset=['date_added','rating','duration'])

In [10]:
df.isnull().sum()

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8790 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   cast          8790 non-null   object
 5   country       8790 non-null   object
 6   date_added    8790 non-null   object
 7   release_year  8790 non-null   int64 
 8   rating        8790 non-null   object
 9   duration      8790 non-null   object
 10  listed_in     8790 non-null   object
 11  description   8790 non-null   object
dtypes: int64(1), object(11)
memory usage: 892.7+ KB


## 4. Fix Data Types

In [12]:
df.date_added.dtype

dtype('O')

In [13]:
df.date_added.head()

0    September 25, 2021
1    September 24, 2021
2    September 24, 2021
3    September 24, 2021
4    September 24, 2021
Name: date_added, dtype: object

In [14]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())

Finding: titles added per year

In [15]:
df.groupby(df.date_added.dt.year).title.count()

date_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      24
2015      82
2016     426
2017    1185
2018    1648
2019    2016
2020    1879
2021    1498
Name: title, dtype: int64

## 5. Split the Duration Column
`duration` mixed minutes (movies) and seasons (TV shows), so it was split into `duration_minutes` and `duration_seasons`. Non-applicable rows are left as NaN rather than 0, because 0 would wrongly suggest a real value.

In [16]:
duration_minutes = df[df.type == 'Movie'].duration.str.split(' ').str[0].astype('int')

In [17]:
duration_seasons = df[df.type == 'TV Show'].duration.str.split(' ').str[0].astype('int')

In [18]:
df['duration_minutes'] = duration_minutes
df['duration_seasons'] = duration_seasons

## 6. Explode Genres and Countries
`listed_in` and `country` hold several comma-separated values in one cell. Splitting and exploding gives one row per genre/country, so counts per category are accurate. Saved as separate DataFrames so the main one keeps one row per title.
For `country`, stray leading/trailing commas were stripped before splitting, because a few entries like ", France, Algeria" otherwise produced blank or malformed country names.

In [19]:
df.listed_in.head(10)

0                                        Documentaries
1      International TV Shows, TV Dramas, TV Mysteries
2    Crime TV Shows, International TV Shows, TV Act...
3                               Docuseries, Reality TV
4    International TV Shows, Romantic TV Shows, TV ...
5                   TV Dramas, TV Horror, TV Mysteries
6                             Children & Family Movies
7     Dramas, Independent Movies, International Movies
8                         British TV Shows, Reality TV
9                                     Comedies, Dramas
Name: listed_in, dtype: object

In [20]:
df_genres = df.assign(listed_in=df.listed_in.str.split(', ')).explode('listed_in')

In [21]:
df_genres.groupby('listed_in').title.count().sort_values(ascending=False)

listed_in
International Movies            2752
Dramas                          2426
Comedies                        1674
International TV Shows          1349
Documentaries                    869
Action & Adventure               859
TV Dramas                        762
Independent Movies               756
Children & Family Movies         641
Romantic Movies                  616
Thrillers                        577
TV Comedies                      573
Crime TV Shows                   469
Kids' TV                         448
Docuseries                       394
Music & Musicals                 375
Romantic TV Shows                370
Horror Movies                    357
Stand-Up Comedy                  343
Reality TV                       255
British TV Shows                 252
Sci-Fi & Fantasy                 243
Sports Movies                    219
Anime Series                     174
Spanish-Language TV Shows        173
TV Action & Adventure            167
Korean TV Shows             

In [22]:
df_country = df.assign(country = df.country.str.strip(', ').str.split(', ')).explode('country')

In [23]:
df_country.groupby('country').title.count().sort_values(ascending=False)

country
United States     3681
India             1046
Unknown            829
United Kingdom     805
Canada             445
                  ... 
Somalia              1
Samoa                1
Sudan                1
Uganda               1
Vatican City         1
Name: title, Length: 123, dtype: int64

## 7. Analysis and Findings
- TV-MA is the most common rating. Its counts rise with the catalog; 2021 is lower because the data stops partway through that year.
- "Unknown" ranks 3rd in country counts because it labels missing data, not a real country.

Finding: movie vs TV show split

In [24]:
df.groupby('type').type.count()

type
Movie      6126
TV Show    2664
Name: type, dtype: int64

In [25]:
df['year'] = df.date_added.dt.year

In [26]:
df.groupby(['year','rating']).title.count()

year  rating
2008  TV-MA       2
2009  TV-MA       2
2010  NR          1
2011  TV-14       5
      TV-MA       3
               ... 
2021  TV-G       44
      TV-MA     489
      TV-PG      97
      TV-Y       57
      TV-Y7      87
Name: title, Length: 106, dtype: int64

In [27]:
df.groupby(['year', 'rating']).title.count().loc[:, 'TV-MA']

year
2008      2
2009      2
2011      3
2013      3
2014     12
2015     29
2016    162
2017    446
2018    650
2019    736
2020    671
2021    489
Name: title, dtype: int64

## 8. Merge IMDb Ratings
The IMDb data had duplicate titles, so ratings were averaged per title before merging (otherwise the merge created extra rows). A left merge keeps all Netflix titles; unmatched ones get NaN.

Limitations:
- IMDb covers only part of the catalog: 6,285 of 8,790 titles have no rating (3,677 of 6,126 movies).
- The IMDb file contains only movies, so the ~56 TV shows that received a rating matched a movie's title by coincidence. These are kept as they are, and TV-genre averages below should be treated with caution.
- The merge matches on title only, so two different films sharing a title would get the same rating.

In [28]:
imdb_df = pd.read_csv("/kaggle/input/datasets/sukhmandeepsinghbrar/netflix-all-movie-ratings-by-imdb/imdb_data.csv")
imdb_df.head(10)

,Rating,Name
0,8.8,Inception\n(2010)
1,8.7,The Matrix\n(1999)
2,9.0,The Lord of the Rings: The Return of the King\...
3,8.8,The Lord of the Rings: The Two Towers\n(2002)
4,8.5,The Departed\n(2006)
5,8.4,Avengers: Infinity War\n(2018)
6,8.4,Raiders of the Lost Ark\n(1981)
7,8.4,Spider-Man: Into the Spider-Verse\n(2018)
8,8.4,Inglourious Basterds\n(2009)
9,8.3,American Beauty\n(1999)


In [29]:
imdb_df_title = imdb_df.Name.str.split("\n").str[0]
imdb_df_title.head(10)

0                                        Inception
1                                       The Matrix
2    The Lord of the Rings: The Return of the King
3            The Lord of the Rings: The Two Towers
4                                     The Departed
5                           Avengers: Infinity War
6                          Raiders of the Lost Ark
7                Spider-Man: Into the Spider-Verse
8                             Inglourious Basterds
9                                  American Beauty
Name: Name, dtype: object

In [30]:
imdb_df['title'] = imdb_df_title

In [31]:
imdb_df.title.duplicated().sum()

np.int64(141)

In [32]:
imdb_df[imdb_df.title.duplicated(keep=False)].sort_values('title').head(20)

,Rating,Name,title
912,7.2,10 Cloverfield Lane\n(2016),10 Cloverfield Lane
121,7.2,10 Cloverfield Lane\n(2016),10 Cloverfield Lane
1560,7.5,100 Meters\n(2016),100 Meters
2657,7.5,100 Meters\n(2016),100 Meters
3614,5.8,2012\n(I) (2009),2012
1250,5.8,2012\n(I) (2009),2012
1554,5.8,2012\n(I) (2009),2012
1254,5.4,A Christmas Prince: The Royal Baby\n(2019),A Christmas Prince: The Royal Baby
575,5.4,A Christmas Prince: The Royal Baby\n(2019),A Christmas Prince: The Royal Baby
1548,5.3,A Cinderella Story: Christmas Wish\n(2019 Video),A Cinderella Story: Christmas Wish


In [33]:
imdb_df[imdb_df.title.duplicated(keep=False)].groupby('title').Rating.nunique().value_counts()

Rating
1    106
2      6
3      1
Name: count, dtype: int64

In [34]:
imdb_df_clean = imdb_df.groupby('title').Rating.mean().reset_index()

In [35]:
merged_df = df.merge(imdb_df_clean, on='title', how='left')
merged_df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_minutes,duration_seasons,year,Rating
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",90.0,NaN,2021,NaN
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",NaN,2.0,2021,NaN
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,NaN,1.0,2021,NaN
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",NaN,1.0,2021,NaN
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,NaN,2.0,2021,NaN


In [36]:
merged_df.shape

(8790, 16)

In [37]:
merged_df.Rating.isnull().sum()

np.int64(6285)

In [38]:
merged_df[merged_df.type == 'Movie'].Rating.isnull().sum()

np.int64(3677)

## 9. Average Rating by Genre
Averages are only reliable where many titles have a rating. TV Thrillers (2 ratings) and Science & Nature TV (1 rating) top the list on almost no data. Among genres with 100+ rated titles, Documentaries (6.88, n=357) and Stand-Up Comedy (6.73, n=189) rate highest, and Horror Movies (5.30, n=162) lowest. Ratings on TV genres come from coincidental title matches with movies, so they are not real show ratings.

In [39]:
merged_df_genres = merged_df.assign(listed_in = merged_df.listed_in.str.split(', ')).explode('listed_in')

In [40]:
merged_df_genres.groupby('listed_in').Rating.mean().sort_values(ascending=False)

listed_in
TV Thrillers                    7.950000
Science & Nature TV             7.900000
TV Horror                       7.666667
Anime Series                    7.600000
TV Dramas                       7.461111
TV Mysteries                    7.440000
TV Sci-Fi & Fantasy             7.350000
Classic Movies                  7.207692
Docuseries                      7.200000
International TV Shows          7.190909
TV Action & Adventure           7.175000
British TV Shows                7.114286
Crime TV Shows                  7.077778
Anime Features                  6.986667
Movies                          6.880000
Documentaries                   6.878151
Korean TV Shows                 6.833333
TV Comedies                     6.800000
Cult Movies                     6.797143
Stand-Up Comedy                 6.729101
Sports Movies                   6.553488
Romantic TV Shows               6.542857
Stand-Up Comedy & Talk Shows    6.500000
TV Shows                        6.450000
LGBTQ 

In [41]:
merged_df_genres.groupby('listed_in').Rating.count().sort_values(ascending=False)

listed_in
International Movies            1045
Dramas                           975
Comedies                         658
Documentaries                    357
Independent Movies               352
Action & Adventure               350
Thrillers                        254
Romantic Movies                  215
Children & Family Movies         196
Stand-Up Comedy                  189
Horror Movies                    162
Music & Musicals                 141
Sci-Fi & Fantasy                 123
Sports Movies                     86
Classic Movies                    39
LGBTQ Movies                      38
Cult Movies                       35
Faith & Spirituality              26
International TV Shows            22
TV Dramas                         18
Kids' TV                          16
Anime Features                    15
TV Comedies                       14
Crime TV Shows                     9
British TV Shows                   7
Romantic TV Shows                  7
TV Mysteries                